In [38]:
import torch
import torchvision
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import os
from torchvision import transforms
from transformers import AutoModel, AutoTokenizer
import torchvision.models as models

# import train

In [39]:
model_name = "google-bert/bert-base-uncased"
class MultiModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.text_model = AutoModel.from_pretrained(model_name) #trust_remote_code=True
        self.image_model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)#, pretrained=True
        self.image_model.fc = nn.Identity()

        total_bert_layers = len(self.text_model.encoder.layer)
        for param in self.text_model.parameters():
            param.requires_grad = False
        bert_unfreez = 1
        for i in range(total_bert_layers - bert_unfreez, total_bert_layers):
            for param in self.text_model.encoder.layer[i].parameters():
                param.requires_grad = True
        for param in self.text_model.pooler.parameters():
            param.requires_grad = True

        for param in self.image_model.parameters():
            param.requires_grad = False
        # for param in self.image_model.layer3.parameters():
        #     param.requires_grad = True
        for param in self.image_model.layer4.parameters():
            param.requires_grad = True

        self.num_classes = num_classes
        text_output = self.text_model.config.hidden_size
        image_output = 512
        comb_out = text_output + image_output
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(comb_out, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, self.num_classes)
        )
    
    def forward(self, input_ids, attention_mask, image):
        text_features = self.text_model(
            input_ids = input_ids, 
            attention_mask = attention_mask
        ).pooler_output

        image_features = self.image_model(image)

        comb = torch.cat([text_features, image_features], dim=1)

        return self.classifier(comb)

In [40]:
checkpoint = torch.load("/kaggle/input/model-v-0/best_model.pth")
# model = train.MultiModel(num_classes=7)
model = MultiModel(num_classes=7)
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [ ]:
def inference(model, text, image_path = ""):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.eval()
    model.to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    text_tokens = tokenizer(
        text,
        max_length = 512, 
        padding = 'max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids = text_tokens['input_ids'].to(device)
    attention_mask = text_tokens['attention_mask'].to(device)

    if image_path != "":
        image = Image.open(image_path).convert("RGB")
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                std=[0.229, 0.224, 0.225])
        ])
        image = transform(image).unsqueeze(0).to(device)
    else:
        image = torch.ones(1, 3, 224, 224).to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask, image)
        prob = torch.softmax(outputs, dim=1)
        _, predict = prob.max(dim=1)

    return predict, prob

In [54]:
ds.labels, ds.label_to_idx

(['acmmisis',
  'aiknowledgeclub',
  'art_klaster',
  'itatmisis',
  'nust_misis',
  'sportmisis',
  'youthmisis'],
 {'acmmisis': 0,
  'aiknowledgeclub': 1,
  'art_klaster': 2,
  'itatmisis': 3,
  'nust_misis': 4,
  'sportmisis': 5,
  'youthmisis': 6})

(['acmmisis',
  'aiknowledgeclub',
  'art_klaster',
  'itatmisis',
  'nust_misis',
  'sportmisis',
  'youthmisis'],
 {'acmmisis': 0,
  'aiknowledgeclub': 1,
  'art_klaster': 2,
  'itatmisis': 3,
  'nust_misis': 4,
  'sportmisis': 5,
  'youthmisis': 6})

In [55]:
text = "Сегодня в университете прошла конференция по искусственному интеллекту"
pred_class, probs = inference(model, text)
pred_class, probs, ds.labels[pred_class]

(tensor([4], device='cuda:0'),
 tensor([[0.0102, 0.0131, 0.0062, 0.0068, 0.8627, 0.0087, 0.0923]],
        device='cuda:0'),
 'nust_misis')

In [56]:
text = "Спортивные соревнования в НИТУ МИСИС"
pred_class, probs = inference(model, text)
ds.labels[pred_class], probs

('nust_misis',
 tensor([[0.0609, 0.1003, 0.0174, 0.0605, 0.6083, 0.0643, 0.0883]],
        device='cuda:0'))

In [57]:
text = "Спортивные соревнования в НИТУ МИСИС"
image_path = "/kaggle/input/misis-posts-classification/processed/images/563_1765028327.jpg"
pred_class, probs = inference(model, text, image_path)
ds.labels[pred_class], probs

('sportmisis',
 tensor([[2.2982e-03, 5.7126e-03, 4.5406e-04, 1.3955e-03, 1.5499e-02, 9.7265e-01,
          1.9871e-03]], device='cuda:0'))